# Mini-Project: Diabetes Prediction & Model Evaluation (Logistic Regression & ROC Curve)

This notebook covers problem understanding, data loading/exploration, standardization, model training, evaluation metrics (Accuracy, Confusion Matrix, Precision, Recall, F1-Score), decision boundary visualization, and ROC Curve plotting.

In [ ]:
# [Cell 2: Code] Import Necessary Libraries
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn import metrics
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print("Libraries imported successfully!")

## Exercise 1: Understanding the Problem and Data Collection

- **Problem Statement**: We want to predict whether an individual has diabetes based on diagnostic measurements (e.g., glucose levels, BMI, age).
- **Tasks**: Load the dataset, check positive and negative cases, and split the data into training and testing sets.

In [ ]:
# [Cell 4: Code] Load and Explore the Diabetes Dataset
# Using the classic Pima Indians Diabetes dataset
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
column_names = [
    'Pregnancies',
    'Glucose',
    'BloodPressure',
    'SkinThickness',
    'Insulin',
    'BMI',
    'DiabetesPedigreeFunction',
    'Age',
    'Outcome',
]
data = pd.read_csv(url, names=column_names)

print("--- First 5 Rows ---")
display(data.head())

print("\n--- Dataset Information ---")
print(data.info())

# Count positive (1) and negative (0) cases
case_counts = data['Outcome'].value_counts()
print(f"\nPositive Cases (Has Diabetes = 1): {case_counts[1]}")
print(f"Negative Cases (No Diabetes = 0): {case_counts[0]}")

# Define features (X) and target (y)
X = data.drop('Outcome', axis=1)
y = data['Outcome']

# Split the dataset into training (70%) and testing (30%) sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)
print(f"\nTraining set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

## Exercise 2: Model Picking and Standardization

- **Model Choice**: **Logistic Regression** is ideal here because it is a binary classification problem (predicting 0 or 1) and provides output probabilities.
- **Standardization**: Yes, features have very different scales (e.g., Age vs. Insulin). We use `StandardScaler()` to center and scale features to mean=0 and variance=1, preventing features with larger numerical magnitudes from dominating the model.

In [ ]:
# [Cell 6: Code] Feature Standardization
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Data standardized successfully!")

## Exercise 3: Model Training

We instantiate and train the Logistic Regression model using the scaled training dataset.

In [ ]:
# [Cell 8: Code] Train Logistic Regression Model
log_regression = LogisticRegression(random_state=42)
log_regression.fit(X_train_scaled, y_train)

# Make predictions on the test set
y_pred = log_regression.predict(X_test_scaled)
y_pred_proba = log_regression.predict_proba(X_test_scaled)[:, 1]

print("Model trained and predictions generated successfully!")

## Exercise 4: Evaluation Metrics

We evaluate the model using Accuracy, Confusion Matrix, Precision, Recall, and F1-Score.

In [ ]:
# [Cell 10: Code] Calculate and Display Evaluation Metrics
accuracy = accuracy_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)
class_report = classification_report(y_test, y_pred, output_dict=True)

print(f"Accuracy Score: {accuracy:.4f}")
print("\nConfusion Matrix:")
print(conf_matrix)

print("\n--- Classification Report ---")
print(f"Precision (Class 1): {class_report['1']['precision']:.4f}")
print(f"Recall (Class 1):    {class_report['1']['recall']:.4f}")
print(f"F1-Score (Class 1):  {class_report['1']['f1-score']:.4f}")

# Optional: Plot Confusion Matrix visually
import seaborn as sns

plt.figure(figsize=(5, 4))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title('Confusion Matrix Heatmap')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

## Exercise 5: Visualizing Model Performance (Decision Boundary)

To visualize how a classifier separates classes, we can reduce dimensions to 2 features (e.g., Glucose and BMI) or plot a 2D decision boundary.

In [ ]:
# [Cell 12: Code] Visualizing Decision Boundary using 2 Key Features (Glucose & BMI)
X_subset = X[['Glucose', 'BMI']]
X_sub_train, X_sub_test, y_sub_train, y_sub_test = train_test_split(
    X_subset, y, test_size=0.3, random_state=42
)

sub_scaler = StandardScaler()
X_sub_train_scaled = sub_scaler.fit_transform(X_sub_train)

sub_model = LogisticRegression(random_state=42)
sub_model.fit(X_sub_train_scaled, y_sub_train)

# Plotting decision boundary
x_min, x_max = (
    X_sub_train_scaled[:, 0].min() - 1,
    X_sub_train_scaled[:, 0].max() + 1,
)
y_min, y_max = (
    X_sub_train_scaled[:, 1].min() - 1,
    X_sub_train_scaled[:, 1].max() + 1,
)
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.1), np.arange(y_min, y_max, 0.1))

Z = sub_model.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, Z, alpha=0.3, cmap=plt.coolwarm)
plt.scatter(
    X_sub_train_scaled[:, 0],
    X_sub_train_scaled[:, 1],
    c=y_sub_train,
    edgecolors='k',
    cmap=plt.coolwarm,
)
plt.title(
    f'Logistic Regression Decision Boundary (Accuracy: {accuracy*100:.2f}%)'
)
plt.xlabel('Standardized Glucose')
plt.ylabel('Standardized BMI')
plt.show()

## Exercise 6: ROC Curve and AUC

We calculate the True Positive Rate (Sensitivity) and False Positive Rate (1 - Specificity), plot the ROC Curve, and calculate the Area Under the Curve (AUC).

In [ ]:
# [Cell 14: Code] Plot ROC Curve and Calculate AUC
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
auc_score = roc_auc_score(y_test, y_pred_proba)

# Create ROC Curve plot
plt.figure(figsize=(8, 6))
plt.plot(
    fpr, tpr, color='blue', label=f'ROC Curve (AUC = {auc_score:.4f})'
)
plt.plot([0, 1], [0, 1], color='red', linestyle='--', label='Random Guess (0.5)')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.xlabel('False Positive Rate (1 - Specificity)')
plt.ylabel('True Positive Rate (Sensitivity)')
plt.legend(loc='lower right')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

print(f'Model AUC Score: {auc_score:.4f}')